# 05 — Đánh giá Test, phân tích lỗi và triển khai
Chỉ đọc bundle/cache từ chuỗi notebook 02–04 cùng run_id. Không fallback sang production.
Mô hình và threshold đã chốt trên Validation; kết quả Test không dùng để chọn lại.


In [5]:
import sys
from pathlib import Path
PROJECT_ROOT=Path.cwd().resolve()
if not (PROJECT_ROOT/'src').is_dir(): PROJECT_ROOT=PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0,str(PROJECT_ROOT))
import numpy as np
import pandas as pd
import scipy.sparse as sp
from IPython.display import display
from src.config import NOTEBOOK_SAVED_MODELS_DIR,NOTEBOOK_METRICS_DIR,NOTEBOOK_FIGURES_DIR,PROCESSED_DATA_DIR,FBETA_BETA
from src.models.persistence import load_spam_pipeline,save_spam_pipeline
from src.evaluation.metrics import SpamModelEvaluator
from src.evaluation.error_analysis import extract_misclassified_samples,profile_error_patterns
from src.evaluation.visualization import plot_confusion_matrix_heatmap,plot_roc_and_pr_curves,plot_models_comparison_bar
from src.deployment.predictor import SpamInferenceService
for directory in (NOTEBOOK_METRICS_DIR,NOTEBOOK_FIGURES_DIR): directory.mkdir(parents=True,exist_ok=True)
pipeline_path=NOTEBOOK_SAVED_MODELS_DIR/'spam_pipeline.joblib'
bundle=load_spam_pipeline(pipeline_path)
assert bundle['run_id']==(PROCESSED_DATA_DIR/'notebook_run_id.txt').read_text(encoding='utf-8').strip(),'Chạy lại 02–04.'
X_test=sp.load_npz(PROCESSED_DATA_DIR/'X_test.npz'); y_test=np.load(PROCESSED_DATA_DIR/'y_test.npy')
test_df=pd.read_csv(PROCESSED_DATA_DIR/'test_clean.csv',keep_default_na=False)
assert test_df.source_id.to_list()==bundle['source_ids']['test'],'Thứ tự dữ liệu Test không khớp artifacts.'
assert np.array_equal((test_df.label=='spam').astype(int).to_numpy(),y_test)
evaluator=SpamModelEvaluator()
model=bundle['model']; best_threshold=bundle['best_threshold']

from hashlib import sha256
if not bundle.get('cache_sha256'):
    raise RuntimeError('Cache thiếu manifest: chạy lại notebook 02–04.')
for filename, expected_hash in bundle['cache_sha256'].items():
    actual_hash=sha256((PROCESSED_DATA_DIR/filename).read_bytes()).hexdigest()
    if actual_hash != expected_hash:
        raise RuntimeError(f'Cache không khớp artifacts: {filename}; chạy lại notebook 02–04.')
print('PASS: hash ma trận, nhãn và CSV khớp artifacts.')


PASS: hash ma trận, nhãn và CSV khớp artifacts.


## 1. Tính lại đầy đủ metrics và biểu đồ từ Test
Bảng so sánh dùng threshold đã chọn riêng cho mỗi ứng viên trên Validation (Hard Voting dùng 0.5).
Winner vẫn là lựa chọn ở notebook 04, kể cả khi ứng viên khác có Test F1 cao hơn.


In [6]:
results={}
for name,candidate in bundle['candidate_models'].items():
    scores=candidate.predict_proba(X_test)[:,1]
    predictions=(scores>=bundle['candidate_thresholds'][name]).astype(int)
    results[name]=evaluator.compute_all_metrics(y_test,predictions,scores,beta=FBETA_BETA)
summary=evaluator.build_comparison_summary(results)
summary['selected_on_validation']=summary['Mô hình']==bundle['model_name']
summary['run_id']=bundle['run_id']
display(summary)
summary.to_csv(NOTEBOOK_METRICS_DIR/'test_models_comparison.csv',index=False)
scores=model.predict_proba(X_test)[:,1]
predictions=(scores>=best_threshold).astype(int)
final_metrics=results[bundle['model_name']]
plot_confusion_matrix_heatmap(final_metrics['confusion_matrix'],model_name=bundle['model_name'],save_path=NOTEBOOK_FIGURES_DIR/'confusion_matrix.png')
plot_roc_and_pr_curves(y_test,scores,best_threshold=best_threshold,model_name=bundle['model_name'],save_path=NOTEBOOK_FIGURES_DIR/'roc_pr_curve.png')
plot_models_comparison_bar(summary,save_path=NOTEBOOK_FIGURES_DIR/'models_comparison.png')
audit=evaluator.audit_evaluation_consistency(y_test,predictions,scores)
display(audit)
assert (audit.Status=='PASS').all()
bundle['test_metrics']=final_metrics
save_spam_pipeline(bundle,pipeline_path)


,Mô hình,Accuracy,Precision,Recall,F1-Score,F-Beta,ROC-AUC,selected_on_validation,run_id
0,Logistic Regression (Tuned),0.967280,0.922078,0.972603,0.946667,0.962060,0.993091,False,4fbe92d417564dc2b635a3a2114716c7
1,Linear SVM,0.965235,0.932886,0.952055,0.942373,0.948158,0.994588,True,4fbe92d417564dc2b635a3a2114716c7
2,Linear SVM (Tuned),0.965235,0.932886,0.952055,0.942373,0.948158,0.994588,False,4fbe92d417564dc2b635a3a2114716c7
3,Soft Voting,0.959100,0.903846,0.965753,0.933775,0.952703,0.993091,False,4fbe92d417564dc2b635a3a2114716c7
4,Hard Voting,0.955010,0.878049,0.986301,0.929032,0.962567,0.977745,False,4fbe92d417564dc2b635a3a2114716c7
5,Logistic Regression,0.944785,0.860606,0.972603,0.913183,0.947931,0.990894,False,4fbe92d417564dc2b635a3a2114716c7
6,Complement Naive Bayes (Tuned),0.938650,0.886667,0.910959,0.898649,0.905995,0.980351,False,4fbe92d417564dc2b635a3a2114716c7
7,Complement Naive Bayes,0.936605,0.885906,0.904110,0.894915,0.900409,0.978853,False,4fbe92d417564dc2b635a3a2114716c7


,Audit Item,Observed,Expected,Status
0,Total Sample Sum Conservation (TP+FP+TN+FN == N),489.00000,489,PASS
1,Accuracy Mathematical Consistency,0.96524,0.96524,PASS
2,"Precision in [0, 1] Bounds",0.93290,0.0 <= P <= 1.0,PASS
3,"Recall in [0, 1] Bounds",0.95210,0.0 <= R <= 1.0,PASS
4,"F1-Score in [0, 1] Bounds",0.94240,0.0 <= F1 <= 1.0,PASS
5,ROC-AUC Valid (if available),0.99460,>= 0.50,PASS


WindowsPath('E:/Hoc_May_Tren_truong/lab_1 spam_classifier/notebooks/results/saved_models/spam_pipeline.joblib')

## 2. Trích xuất FP/FN và chẩn đoán định tính
Tự tạo CSV từ dự đoán hiện tại; nguyên nhân từ khóa chỉ là giả thuyết, không phải giải thích nhân quả của mô hình.


In [7]:
fp,fn=extract_misclassified_samples(test_df,y_test,predictions,scores)
errors=profile_error_patterns(fp,fn)
if errors.empty:
    errors=pd.concat([fp,fn],ignore_index=True)
    errors['suspected_cause']=pd.Series(dtype=str)
errors['run_id']=bundle['run_id']
errors.to_csv(NOTEBOOK_METRICS_DIR/'misclassified_samples.csv',index=False,encoding='utf-8-sig')
display(pd.DataFrame([{'type':'FP','count':len(fp)},{'type':'FN','count':len(fn)}]))
for kind in ('False Positive','False Negative'):
    subset=errors[errors.error_type.str.contains(kind,na=False)]
    display(subset[['source_id','text','spam_probability','suspected_cause']].head(5))
    if len(subset):
        print(kind,'số từ trung bình:',subset.text.str.split().str.len().mean())
        print(subset.suspected_cause.value_counts().head(5))
print(f"Winner đã chốt: {bundle['model_name']}; Test FP={len(fp)}, FN={len(fn)}, Recall={final_metrics['recall']:.4f}, F1={final_metrics['f1']:.4f}.")
print('Không dùng phân tích lỗi Test để chọn lại ngưỡng hoặc mô hình trong lần đánh giá này.')


,type,count
0,FP,10
1,FN,7


,source_id,text,spam_probability,suspected_cause
0,1823,Subject: fw : playgroup & young families churc...,0.6706,Dấu chấm than (3)
1,1940,Subject: discussion january 10\r\nwe ' re baaa...,0.6300,Dấu chấm than (2)
2,3816,Subject: sitara availability - status update\r...,0.5664,Đặc trưng từ vựng trùng lặp với tập Spam
3,4747,Subject: is this fri feb 11 a problem for taki...,0.6007,Đặc trưng từ vựng trùng lặp với tập Spam
4,2504,Subject: life in general\r\ngood god - - - - w...,0.6576,Đặc trưng từ vựng trùng lặp với tập Spam


False Positive số từ trung bình: 176.4
suspected_cause
Đặc trưng từ vựng trùng lặp với tập Spam    7
Dấu chấm than (3)                           1
Dấu chấm than (2)                           1
Dính từ khóa nhạy cảm (free, call)          1
Name: count, dtype: int64


,source_id,text,spam_probability,suspected_cause
10,3168,Subject: rubbed his hands\r\nfind partners for...,0.5435,"Spam tinh vi, không dùng từ khóa lộ liễu"
11,849,Subject: sample movies\r\nunlimited access to ...,0.5096,Xác suất spam dưới ngưỡng quyết định
12,3642,Subject: whats the word . order your prescr - ...,0.5362,"Spam tinh vi, không dùng từ khóa lộ liễu"
13,2793,"Subject: \r\nmon , 24 may 2004 12 : 14 : 06 - ...",0.3983,Xác suất spam dưới ngưỡng quyết định
14,3817,Subject: important online banking alert\r\ndea...,0.5334,"Spam tinh vi, không dùng từ khóa lộ liễu"


False Negative số từ trung bình: 264.57142857142856
suspected_cause
Spam tinh vi, không dùng từ khóa lộ liễu    4
Xác suất spam dưới ngưỡng quyết định        3
Name: count, dtype: int64
Winner đã chốt: Linear SVM; Test FP=10, FN=7, Recall=0.9521, F1=0.9424.
Không dùng phân tích lỗi Test để chọn lại ngưỡng hoặc mô hình trong lần đánh giá này.


## 3. Kiểm tra parity sau lưu/nạp và demo email mới
Đối chiếu cache với đặc trưng tái tạo trên 10 mẫu Test; xác suất và quyết định phải khớp trước/sau lưu.


In [8]:
service = SpamInferenceService(bundle_path=pipeline_path).load_service()
sample = test_df.head(10)
rebuilt = service.feature_builder.transform(sample).to_scipy_csr()[:, bundle['selected_indices']]
np.testing.assert_allclose(rebuilt.toarray(), X_test[:len(sample)].toarray(), atol=1e-12)
for i, text in enumerate(sample.text):
    result = service.predict_single(text)
    assert result['spam_score'] == round(float(scores[i]), 4)
    assert result['spam_probability'] == round(float(scores[i]), 4)
    assert result['is_spam'] == bool(predictions[i])
    assert result['is_calibrated'] is False
print('PASS: cache, thứ tự Top-K và suy luận sau lưu/nạp nhất quán.')
demo_emails = [
    'URGENT! You won a cash prize! Call now to claim your award!',
    'Dear team, please review the weekly project report and meeting notes.',
    'Free entry to win tickets! Text now to claim your prize!',
    'Hey, are we meeting for lunch today?',
    'Your account statement is ready. Please sign in through the official website.',
]
display(service.predict_batch(demo_emails))
print('Demo minh họa, không phải tập kiểm thử có nhãn. Tỷ lệ phiếu Hard Voting và sigmoid SVM chưa được calibration.')


PASS: cache, thứ tự Top-K và suy luận sau lưu/nạp nhất quán.


,raw_text,prediction,is_spam,spam_score,score_kind,is_calibrated,score_note,spam_probability,threshold_used,risk_level,risk_basis,detected_signals
0,URGENT! You won a cash prize! Call now to clai...,SPAM,True,0.6495,uncalibrated_score,False,Điểm phân loại chưa calibration; không diễn gi...,0.6495,0.555,Cao,"Quy tắc theo ngưỡng phân loại, không phải ước ...","[Từ khóa nhạy cảm: call, cash, claim, prize, u..."
1,"Dear team, please review the weekly project re...",NOT SPAM / HAM,False,0.4529,uncalibrated_score,False,Điểm phân loại chưa calibration; không diễn gi...,0.4529,0.555,Thấp,"Quy tắc theo ngưỡng phân loại, không phải ước ...",[]
2,Free entry to win tickets! Text now to claim y...,SPAM,True,0.7070,uncalibrated_score,False,Điểm phân loại chưa calibration; không diễn gi...,0.7070,0.555,Cao,"Quy tắc theo ngưỡng phân loại, không phải ước ...","[Từ khóa nhạy cảm: claim, free, prize, win, Dấ..."
3,"Hey, are we meeting for lunch today?",SPAM,True,0.6121,uncalibrated_score,False,Điểm phân loại chưa calibration; không diễn gi...,0.6121,0.555,Cao,"Quy tắc theo ngưỡng phân loại, không phải ước ...",[]
4,Your account statement is ready. Please sign i...,SPAM,True,0.6025,uncalibrated_score,False,Điểm phân loại chưa calibration; không diễn gi...,0.6025,0.555,Cao,"Quy tắc theo ngưỡng phân loại, không phải ước ...",[]


Demo minh họa, không phải tập kiểm thử có nhãn. Tỷ lệ phiếu Hard Voting và sigmoid SVM chưa được calibration.
